In [1]:
import sys
import pandas as pd
from PyQt5.QtWidgets import QApplication, QWidget, QPushButton, QVBoxLayout, QLabel, QFormLayout, QLineEdit, QDialog, QDialogButtonBox
from epwgen_methods import run_individual_location

class MainWindow(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("EPWgen")

        # Create layout and buttons
        layout = QVBoxLayout()
        self.button_individual = QPushButton("Retrieve weather data for an individual location")
        self.button_csv = QPushButton("Retrieve data from CSV file list")

        # Connect buttons to their respective actions
        self.button_individual.clicked.connect(self.open_individual_dialog)
        self.button_csv.clicked.connect(self.run_csv_list)

        # Add buttons to the layout
        layout.addWidget(self.button_individual)
        layout.addWidget(self.button_csv)

        self.setLayout(layout)

    def open_individual_dialog(self):
        # Create dialog for input
        dialog = IndividualLocationDialog(self)
        if dialog.exec_() == QDialog.Accepted:
            lat, lon, save_name, year, save_folder, file_type = dialog.get_values()

            # Call the function to retrieve weather data
            retrieve_status, distance, wmo, hdd, cdd, _ = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
            
            # Display the result in a simple message
            result_dialog = QDialog(self)
            result_dialog.setWindowTitle("Weather Data Result")
            result_layout = QVBoxLayout()
            result_layout.addWidget(QLabel(f"Status: {retrieve_status}\nDistance: {distance}\nWMO: {wmo}\nHDD: {hdd}\nCDD: {cdd}"))
            result_dialog.setLayout(result_layout)
            result_dialog.exec_()

    def run_csv_list(self):
        save_folder = 'epws_wmo'
        file_type = 'AMY'
        year = 2022
        csv_list_name = 'resources/zip_code_list.csv'

        # Load the zip codes CSV file
        zipcodes = pd.read_csv(csv_list_name, dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

        # Initialize a counter for iterations
        counter = 0

        for index, row in zipcodes.iterrows():
            print(index)
            zip_code = str(row['zip0']).zfill(5)
            lat = row['lat']
            lon = row['lng']
            save_name = None

            # Retrieve data for the current location
            retrieve_status, distance, wmo, hdd, cdd, _ = run_individual_location(lat, lon, year, file_type, save_folder, save_name)

            # Update the DataFrame
            zipcodes.at[index, f"Do we have data for {year}?"] = retrieve_status
            zipcodes.at[index, f"distance_location_station_miles_{year}"] = distance * 0.000621371
            zipcodes.at[index, f"weather_station_wmo_{year}"] = wmo
            zipcodes.at[index, f"hdd_base65F_{year}"] = hdd
            zipcodes.at[index, f"cdd_base65F_{year}"] = cdd

            counter += 1
            if counter % 10 == 0:
                zipcodes.to_csv(csv_list_name, index=False)

        zipcodes.to_csv(csv_list_name, index=False)

class IndividualLocationDialog(QDialog):
    def __init__(self, parent=None):
        super().__init__(parent)

        self.setWindowTitle("Input Location Details")
        self.form_layout = QFormLayout()

        # Default values
        self.lat_input = QLineEdit("41")
        self.lon_input = QLineEdit("-110")
        self.save_name_input = QLineEdit("tEsT")
        self.year_input = QLineEdit("2022")
        self.save_folder_input = QLineEdit("epws_wmo")
        self.file_type_input = QLineEdit("AMY")

        # Add widgets to form layout
        self.form_layout.addRow("Latitude:", self.lat_input)
        self.form_layout.addRow("Longitude:", self.lon_input)
        self.form_layout.addRow("Save Name:", self.save_name_input)
        self.form_layout.addRow("Year:", self.year_input)
        self.form_layout.addRow("Save Folder:", self.save_folder_input)
        self.form_layout.addRow("File Type:", self.file_type_input)

        self.button_box = QDialogButtonBox(QDialogButtonBox.Ok | QDialogButtonBox.Cancel)
        self.button_box.accepted.connect(self.accept)
        self.button_box.rejected.connect(self.reject)

        # Set layout
        layout = QVBoxLayout()
        layout.addLayout(self.form_layout)
        layout.addWidget(self.button_box)
        self.setLayout(layout)

    def get_values(self):
        return (
            float(self.lat_input.text()),
            float(self.lon_input.text()),
            self.save_name_input.text(),
            int(self.year_input.text()),
            self.save_folder_input.text(),
            self.file_type_input.text()
        )

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec_())


2024-09-09 12:00:20.359 Python[20768:18745423] WARNING: Secure coding is not enabled for restorable state! Enable secure coding by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState: and returning YES.


KeyboardInterrupt: 

SystemExit: 0